<a href="https://colab.research.google.com/github/junseo25/ds2002-fa26/blob/main/notebooks/05-etl-walmart/2026_10_14_Jun_Seo_Lee__%E2%80%94_Mini_ETL_on_Walmart_Sample_%E2%80%94_Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Mini-ETL on Walmart Sample

**Studio — 2026-10-14 · Fall 2026**  
**Class time:** 45 minutes

---

## The midterm, in miniature

Twelve rows of the real thing. Every problem in this sample is in the full dataset you get Monday, just at a scale where you can still see the answer.

You are writing the whole pipeline as **functions**, not as loose cells. That is not stylistic — Monday you will run the same logic on a much larger file, and a pipeline built from functions is one you can re-run and hand to a teammate. A pipeline built from cells in a specific click order is not.

The extract stage is worked below as an example. Transform and load are yours to build.

In [12]:
import pandas as pd, sqlite3
from io import StringIO

RAW = '''transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
TXN1,FL-105,2024-09-06 10:06:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,2,$4.11
TXN2,fl-105,09/06/2024 10:40,PT-12,pop-tarts strawberry,breakfast,1,3.86
TXN3,FL-239,2024-09-06T11:00:00,WATER-24,Bottled Water,Grocery,-1,5.00
TXN1,FL-105,2024-09-06 10:06:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,2,$4.11
TXN4,FL 239,09/06/2024 11:30,SKU#459812,Strawberry Pop-Tart,Breakfast,3,$4.11
TXN5,fl-239,2024-09-06T12:15:00,WATER-24,bottled water,grocery,4,$5.00
TXN6,FL-105,2024-09-07 08:02:00,BTL-WATER-24,Bottled Water 24pk,Grocery,6,5.00
TXN7,FL-105,2024-09-07T09:45:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,5,4.11
TXN8,FL-239,,PT-12,pop-tarts strawberry,Breakfast,2,$4.11
TXN9,FL-105,2024-09-07 14:20:00,FLASH-AA,Flashlight AA,Hardware,NULL,12.99
TXN10,FL-239,2024-09-07T15:00:00,POPTART-BLUE,Pop-Tarts Blueberry,Breakfast,1,4.11
TXN11,FL-105,09/07/2024 16:10,SKU#459812,Strawberry Pop-Tart,breakfast,4,4.11'''

### Extract — worked example

The extract function does one job: get the raw data in, and report what arrived. It does **not** clean anything. Keeping the stages separate is what lets you answer "was this a source problem or did I do it?" later.

In [13]:
def extract():
    """Read the raw transactions. No cleaning here -- report only."""
    df = pd.read_csv(StringIO(RAW))
    print(f'extracted {len(df)} rows, {len(df.columns)} columns')
    print('dtypes that arrived as text:',
          [c for c in df.columns if df[c].dtype == object])
    print('nulls:', df.isnull().sum().sum())
    print('exact duplicates:', df.duplicated().sum())
    return df

raw = extract()
raw.head()

extracted 12 rows, 8 columns
dtypes that arrived as text: ['transaction_id', 'store_id', 'timestamp', 'sku', 'product_name', 'category', 'unit_price']
nulls: 2
exact duplicates: 1


,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
0,TXN1,FL-105,2024-09-06 10:06:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,2.0,$4.11
1,TXN2,fl-105,09/06/2024 10:40,PT-12,pop-tarts strawberry,breakfast,1.0,3.86
2,TXN3,FL-239,2024-09-06T11:00:00,WATER-24,Bottled Water,Grocery,-1.0,5.00
3,TXN1,FL-105,2024-09-06 10:06:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,2.0,$4.11
4,TXN4,FL 239,09/06/2024 11:30,SKU#459812,Strawberry Pop-Tart,Breakfast,3.0,$4.11


Note what the report already told us: `quantity` and `unit_price` both arrived as text, there is a null, and there is a duplicate. That is your transform to-do list, generated from the data instead of from guessing.

**TODO:** before writing any transform, list the specific problems you can see in these twelve rows. There are at least six.

1. _..._
2. _..._
3. _..._
4. _..._
5. _..._
6. _..._

### Build 1 — Transform: types and duplicates

**TODO:** write `transform_types(df)` that returns a new frame with:

- exact duplicate rows removed
- `unit_price` as a float
- `quantity` numeric, with a stated decision about the null and the negative
- `store_id` normalized — note the four spellings of the same two stores
- `timestamp` as a real datetime, coercing the empty one

It must not modify the frame passed in. Print a before/after row count.

In [14]:
def transform_types(df):
    out = df.drop_duplicates().copy()

    out['unit_price'] = (out['unit_price'].astype(str)
                         .str.replace('$', '', regex=False)
                         .str.strip().astype(float))

    out['quantity'] = pd.to_numeric(out['quantity'], errors='coerce')
    # Decision: drop the unparseable quantity and the return. We are measuring
    # units that left the shelf ahead of the storm, so a return is not that.
    out = out[out['quantity'].notna() & (out['quantity'] > 0)].copy()
    out['quantity'] = out['quantity'].astype(int)

    # FL-105 / fl-105 / 'FL 239' / fl-239 all collapse to FL-105 / FL-239
    out['store_id'] = (out['store_id'].str.upper().str.strip()
                       .str.replace(r'[^A-Z0-9]', '', regex=True)
                       .apply(lambda s: f'{s[:2]}-{s[2:]}'))

    out['category'] = out['category'].str.strip().str.title()
    out['timestamp'] = pd.to_datetime(out['timestamp'],
                                      errors='coerce', format='mixed')
    return out

typed = transform_types(raw)
print('rows:', len(raw), '->', len(typed))
print('stores:', sorted(typed['store_id'].unique()))
print('unparseable timestamps:', typed['timestamp'].isna().sum())
typed.dtypes

rows: 12 -> 9
stores: ['FL-105', 'FL-239']
unparseable timestamps: 1


,0
transaction_id,object
store_id,object
timestamp,datetime64[ns]
sku,object
product_name,object
category,object
quantity,int64
unit_price,float64


### Build 2 — Transform: consolidate the products

Four SKUs are strawberry Pop-Tarts. One is blueberry — leave it alone, and be ready to say why. Two are the same bottled water.

**TODO:** write `consolidate(df)` that adds a `canonical_sku` column. Any SKU not in your mapping should keep its original value, not become `NaN`.

Then print units per canonical SKU and confirm the total quantity did not change — consolidation renames things, it never adds or removes units.

In [15]:
CANON = {
    'POPTART-STRAW': 'POPTART-STRAW',
    'PT-12': 'POPTART-STRAW',
    'SKU#459812': 'POPTART-STRAW',
    'WATER-24': 'WATER-24',
    'BTL-WATER-24': 'WATER-24',
}

def consolidate(df):
    out = df.copy()
    out['canonical_sku'] = out['sku'].map(CANON).fillna(out['sku'])
    return out

consolidated = consolidate(typed)
assert consolidated['quantity'].sum() == typed['quantity'].sum()
print(consolidated.groupby('canonical_sku')['quantity'].sum().sort_values(ascending=False))

# TODO: assert total quantity is unchanged from `typed`
# TODO: print units per canonical_sku

canonical_sku
POPTART-STRAW    17
WATER-24         10
POPTART-BLUE      1
Name: quantity, dtype: int64


**Why blueberry stays separate:** _..._

### Build 3 — Transform: derive what the analysis needs

**TODO:** add `revenue` (quantity times unit price) and a `date` column from the timestamp. The `date` column is what you will join weather onto next week, so it needs to be a date — not a full timestamp, not a string.

In [16]:
def derive(df):
    out = df.copy()
    out['revenue'] = (out['quantity'] * out['unit_price']).round(2)
    out['date'] = out['timestamp'].dt.normalize()
    return out

enriched = derive(consolidated)
enriched[['transaction_id', 'canonical_sku', 'quantity', 'revenue', 'date']]

,transaction_id,canonical_sku,quantity,revenue,date
0,TXN1,POPTART-STRAW,2,8.22,2024-09-06
1,TXN2,POPTART-STRAW,1,3.86,2024-09-06
4,TXN4,POPTART-STRAW,3,12.33,2024-09-06
5,TXN5,WATER-24,4,20.00,2024-09-06
6,TXN6,WATER-24,6,30.00,2024-09-07
7,TXN7,POPTART-STRAW,5,20.55,2024-09-07
8,TXN8,POPTART-STRAW,2,8.22,NaT
10,TXN10,POPTART-BLUE,1,4.11,2024-09-07
11,TXN11,POPTART-STRAW,4,16.44,2024-09-07


### Build 4 — Load, and verify the load

**TODO:** write `load(df, conn)` that writes the clean frame to a `clean_sales` table, then reads back the row count and asserts it matches. Writing without verifying is how you discover at 11pm that `if_exists='append'` doubled your table.

In [17]:
def load(df, conn, table='clean_sales'):
    df.to_sql(table, conn, index=False, if_exists='replace')
    written = pd.read_sql_query(f'SELECT COUNT(*) AS n FROM {table}', conn)['n'][0]
    assert written == len(df), f'wrote {written}, expected {len(df)}'
    print(f'loaded {written} rows into {table}')
    return written

conn = sqlite3.connect(':memory:')
load(enriched, conn)

pd.read_sql_query(
    'SELECT canonical_sku, SUM(quantity) AS units, ROUND(SUM(revenue), 2) AS revenue '
    'FROM clean_sales GROUP BY canonical_sku ORDER BY units DESC', conn)

loaded 9 rows into clean_sales


,canonical_sku,units,revenue
0,POPTART-STRAW,17,69.62
1,WATER-24,10,50.00
2,POPTART-BLUE,1,4.11


### Build 5 — run the whole thing as one pipeline

**TODO:** wire the stages together in one function, then run it twice against a fresh connection and confirm you get identical results. That is the idempotency property from Monday, tested rather than assumed.

In [18]:
def run_pipeline(conn):
    df = extract()
    df = transform_types(df)
    df = consolidate(df)
    df = derive(df)
    load(df, conn)
    return df

first = run_pipeline(sqlite3.connect(':memory:'))
second = run_pipeline(sqlite3.connect(':memory:'))
# TODO: assert the two runs produced the same shape and the same revenue total

assert first.shape == second.shape
assert first['revenue'].sum() == second['revenue'].sum()
print('identical across runs:', first.equals(second))

extracted 12 rows, 8 columns
dtypes that arrived as text: ['transaction_id', 'store_id', 'timestamp', 'sku', 'product_name', 'category', 'unit_price']
nulls: 2
exact duplicates: 1
loaded 9 rows into clean_sales
extracted 12 rows, 8 columns
dtypes that arrived as text: ['transaction_id', 'store_id', 'timestamp', 'sku', 'product_name', 'category', 'unit_price']
nulls: 2
exact duplicates: 1
loaded 9 rows into clean_sales
identical across runs: True


### Build 6 — the question the pipeline exists to answer

**TODO:** did strawberry Pop-Tart sales rise from Sept 6 to Sept 7? Print units by date for your canonical strawberry SKU, and say whether two days is enough evidence to conclude anything.

That second part matters. On the midterm you will have a real baseline to compare against; here you do not, and saying so is the correct answer.

In [19]:
# TODO
straw = enriched[enriched['canonical_sku'] == 'POPTART-STRAW']
by_day = straw.groupby(straw['date'].dt.date)['quantity'].sum()
print(by_day)
print()
print('units with no date:', int(straw.loc[straw['date'].isna(), 'quantity'].sum()))

date
2024-09-06    6
2024-09-07    9
Name: quantity, dtype: int64

units with no date: 2


---

## Checkpoint (participation)

Report your row count after transform, your canonical SKU count, and whether running the pipeline twice gave the same revenue total.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [21]:
# Checkpoint
rows_after_transform = 9   # TODO
canonical_skus = 3         # TODO: how many distinct products you ended with
idempotent = True             # TODO: True/False from Build 5
blueberry_call = 'As blueberry poptarts are a completely different product than strawberry poptarts. If we combined them, it would dramatically change the results we get when we are trying to specifically find information about strawberry poptarts instead of poptarts in general.'       # TODO: why you kept blueberry separate

print('rows after transform:', rows_after_transform)
print('canonical SKUs:', canonical_skus)
print('same result twice:', idempotent)
print('blueberry:', blueberry_call)

rows after transform: 9
canonical SKUs: 3
same result twice: True
blueberry: As blueberry poptarts are a completely different product than strawberry poptarts. If we combined them, it would dramatically change the results we get when we are trying to specifically find information about strawberry poptarts instead of poptarts in general.
